# 1. Data, splits and graph

What the data looks like, how it is split in time, and the graph that is built from it. If this says *fixture*, the numbers describe generated data and are **not findings**.

In [1]:
%matplotlib inline
from pipeline.notebook_support import *
import pandas as pd
import numpy as np
pd.set_option('display.width', 160)
prep = get_prepared()
cfg, df = prep.cfg, prep.df
print('FIXTURE DATA - not real results' if fixture_mode() else 'IEEE-CIS')

C:\Users\User\.venvs\unified_risk_engine\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\.venvs\unified_risk_engine\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\User\OneDrive\Documents\Coding Projects\unified-risk-engine\pipeline\graph_builder.py:27: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pyto

IEEE-CIS


In [2]:
days = (df['TransactionDT'].max() - df['TransactionDT'].min()) / 86400
pd.Series({'transactions': len(df), 'users': df['user'].nunique(), 'days': round(days, 1),
           'fraud rate': round(df['isFraud'].mean(), 4)})

transactions    590540.000
users            39974.000
days               182.000
fraud rate           0.035
dtype: float64

## Time periods

Train, validation and test are consecutive in time, separated by an embargo so that labels that arrive late cannot leak across a boundary.

In [3]:
from pipeline.splits import DAY, period_of
p = prep.periods
period = pd.Series(period_of(df['TransactionDT'].to_numpy(), p), name='period')
table = df.groupby(period).agg(rows=('isFraud', 'size'), fraud_rate=('isFraud', 'mean'))
table

,rows,fraud_rate
period,,
embargo,45107,0.032257
test,66606,0.036153
train,413378,0.035169
val,65449,0.034561


In [4]:
gaps = {'train->validation (days)': (p.val_start - p.t_train) / DAY, 'validation->test (days)': (p.test_start - p.val_end) / DAY}
assert all(abs(g - p.embargo / DAY) < 1e-6 for g in gaps.values()), gaps
gaps

{'train->validation (days)': 7.0, 'validation->test (days)': 7.0}

## Account-risk population

A proxy label from fraud history up to the end of training, for users with enough history.

In [5]:
acc = prep.account
pd.DataFrame({'users': acc.groupby('split').size(), 'eligible': acc[acc['eligible']].groupby('split').size(),
              'positives': acc[acc['eligible']].groupby('split')['label'].sum()})

,users,eligible,positives
split,,,
test,5996,2342,112
train,27982,10745,477
val,5996,2270,110


## Threshold choice

The account label's fraud-rate threshold is chosen by how many positives it leaves, never by model performance. The configured value is shown against alternatives.

In [6]:
rates = acc[acc['eligible']]['fraud_rate']
print('configured threshold:', cfg['account']['fraud_rate_threshold'])
pd.DataFrame({t: {'positives': int((rates >= t).sum()), 'prevalence': round(float((rates >= t).mean()), 4)}
              for t in (0.1, 0.2, 0.3, 0.5)}).T

configured threshold: 0.2


,positives,prevalence
0.1,1072.0,0.0698
0.2,699.0,0.0455
0.3,473.0,0.0308
0.5,283.0,0.0184


## Leakage checks

Train-period fraud rows come only from train users and end by the end of the training period.

In [7]:
tr = prep.graph['transaction'].train_mask.numpy()
assert (df.loc[tr, 'TransactionDT'] <= p.t_train).all()
assert (prep.user_split.reindex(df.loc[tr, 'user']).to_numpy() == 'train').all()
print('leakage checks passed')

leakage checks passed


## Graph

Node and edge counts of the heterogeneous graph (transaction, user, email_domain, device).

In [8]:
g = prep.graph
display(pd.DataFrame({'nodes': {nt: g[nt].num_nodes for nt in g.node_types}}))
display(pd.DataFrame({'edges': {str(et): g[et].edge_index.shape[1] for et in g.edge_types}}))

,nodes
transaction,590540
user,39974
email_domain,59
device,1786


,edges
"('user', 'owns', 'transaction')",590540
"('transaction', 'rev_owns', 'user')",590540
"('transaction', 'uses_email', 'email_domain')",496084
"('email_domain', 'rev_uses_email', 'transaction')",496084
"('transaction', 'uses_device', 'device')",118666
"('device', 'rev_uses_device', 'transaction')",118666
